# Experiment: Deterministic Drift Check With Core `target_diffusion`

Objective:
- Replace the old public `target_vx` framing with the deterministic continuous-drive limit `dx = v \, dt`, `c = 0`.
- Use the core `target_diffusion` calibration path and the model's own continuous decision-space reference path `x_traj`.
- Verify that the decoded circuit state overlaps the expected linear reference trajectory for small constant drift.

Success criteria:
- The notebook calibrates `c_{BE}(\theta)` through `prepare_target_diffusion_mode()`.
- For one representative small drift, `x_ref`, `x_E`, and `x_B` are visually close and nearly linear after onset.
- A small drift sweep shows decoded slope tracking the target latent slope in the weak-drive regime.


In [1]:
from __future__ import annotations

import runpy
from pathlib import Path

_NOTEBOOK_SETUP = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    helper = candidate / "figures_code" / "notebook_setup.py"
    if helper.exists():
        _NOTEBOOK_SETUP = runpy.run_path(str(helper))
        break
if _NOTEBOOK_SETUP is None:
    raise RuntimeError("Could not locate figures_code/notebook_setup.py")

REPO_ROOT = _NOTEBOOK_SETUP["REPO_ROOT"]

import json
import os
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path("/tmp") / "mpl_deterministic_drift_check")

import matplotlib.pyplot as plt
import numpy as np

from CANN_DDM_model_rate_based import CANN_DDM_model
from rate_model_core.default_params import build_stable_default_params

SEED = 4
DT_MS = 1.0
T_START = 20
DUR = 300
THETA_MARGIN = 0.02
C_BE_SWEEP = np.array([0.0, 0.02, 0.04, 0.06, 0.08], dtype=float)
MIN_ACCUMULATION_SAMPLES = 80
C_TARGET = 0.0
V_TARGET = 0.10
V_SWEEP = np.array([0.02, 0.05, 0.08, 0.10, 0.12], dtype=float)
FIT_WINDOW = (80, 220)

np.random.seed(SEED)
REPO_ROOT


PosixPath('/projectnb/ecog-eeg/cyw6/CANN_DDM_rate_model')

## Notebook choices

This notebook mirrors the Brownian quick check, but in the deterministic limit.

- It uses `decision_mode="continuous"`.
- It uses the core `target_diffusion` calibration path.
- It sets `noise_scale = 0` and uses only constant latent drift `v`.
- It reads `x_ref` directly from `model.x_traj`.


In [2]:
def build_baseline_params(*, c: float, v: float, x0: float = 0.5, seed: int = SEED, kappa: float | None = None) -> dict:
    params = build_stable_default_params()
    params["bump_pop"]["noise_scale_bump"] = 0.0
    params["edge_pop"]["noise_scale_edge"] = 0.0

    c_be_params = {
        "mode": "target_diffusion",
        "theta_margin": THETA_MARGIN,
    }
    if kappa is not None:
        c_be_params["kappa"] = float(kappa)
    params["bump_pop"]["c_BE_params"] = c_be_params

    params["decision_space_params"]["decision_mode"] = "continuous"
    params["decision_space_params"]["t_start"] = T_START
    params["decision_space_params"]["dur"] = DUR
    params["decision_space_params"]["dt_DDM"] = DT_MS
    params["decision_space_params"]["x0"] = float(x0)
    params["decision_space_params"]["seed"] = int(seed)
    params["decision_space_params"]["drift_rate"] = float(v)
    params["decision_space_params"]["noise_scale"] = float(c)
    return params


def make_calibrated_model(*, seed: int = SEED):
    model = CANN_DDM_model(CANN_params=build_baseline_params(c=C_TARGET, v=V_TARGET, seed=seed))
    result = model.prepare_target_diffusion_mode(
        c_be_sweep=C_BE_SWEEP,
        min_accumulation_samples=MIN_ACCUMULATION_SAMPLES,
    )
    return model, result


def make_prepared_model(*, v: float, seed: int, kappa: float):
    return CANN_DDM_model(CANN_params=build_baseline_params(c=C_TARGET, v=v, seed=seed, kappa=kappa))


def estimate_slope(times: np.ndarray, values: np.ndarray, window: tuple[int, int]) -> float:
    start, stop = window
    mask = (times >= start) & (times <= stop)
    if np.count_nonzero(mask) < 2:
        return float("nan")
    slope, _ = np.polyfit(times[mask], values[mask], deg=1)
    return float(slope)


def run_trial(*, v: float, seed: int, kappa: float):
    model = make_prepared_model(v=v, seed=seed, kappa=kappa)
    runner = model.run_simulation(
        mon_vars=["x_E", "x_B", "theta_E", "theta_B", "v_drift", "v_drive", "hit_boundary"],
        progress_bar=False,
        dt=DT_MS,
        get_RT=False,
    )
    times = np.arange(DUR, dtype=float)
    x_ref = np.asarray(model.x_traj, dtype=float)
    x_e = np.asarray(runner.mon.x_E).reshape(-1)
    x_b = np.asarray(runner.mon.x_B).reshape(-1)
    theta_e = np.asarray(runner.mon.theta_E).reshape(-1)
    theta_b = np.asarray(runner.mon.theta_B).reshape(-1)
    v_drift = np.asarray(runner.mon.v_drift).reshape(-1)
    v_drive = np.asarray(runner.mon.v_drive).reshape(-1)
    hit = np.asarray(runner.mon.hit_boundary).reshape(-1)
    return {
        "v_target": float(v),
        "times": times,
        "x_ref": x_ref,
        "x_E": x_e,
        "x_B": x_b,
        "theta_E": theta_e,
        "theta_B": theta_b,
        "v_drift": v_drift,
        "v_drive": v_drive,
        "ref_slope": estimate_slope(times, x_ref, FIT_WINDOW),
        "edge_slope": estimate_slope(times, x_e, FIT_WINDOW),
        "bump_slope": estimate_slope(times, x_b, FIT_WINDOW),
        "mean_abs_error_xE": float(np.mean(np.abs(x_e[T_START:] - x_ref[T_START:]))),
        "mean_abs_error_xB": float(np.mean(np.abs(x_b[T_START:] - x_ref[T_START:]))),
        "alignment_mean_abs": float(np.mean(np.abs(theta_b[T_START:] - theta_e[T_START:]))),
        "hit_boundary": bool(np.any(hit)),
    }


## Step 1: calibrate the shared `c_{BE}(\theta)` profile

The deterministic and stochastic checks now share the same core calibration path.


In [3]:
calibration_model, calibration_result = make_calibrated_model()
sweep_results = calibration_result["sweep_results"]
c_be_vals = np.array([row["c_BE"] for row in sweep_results], dtype=float)
v_theta_vals = np.array([row["v_theta_E"] for row in sweep_results], dtype=float)
valid_mask = np.array([row["valid"] for row in sweep_results], dtype=bool)

theta_grid = np.asarray(calibration_result["theta_grid"], dtype=float)
c_be_theta = np.asarray(calibration_result["c_be_theta"], dtype=float)

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax[0].plot(c_be_vals, v_theta_vals, "o-", color="#1d52a1")
ax[0].scatter(c_be_vals[valid_mask], v_theta_vals[valid_mask], color="#059669", label="valid")
ax[0].set_xlabel("constant c_BE")
ax[0].set_ylabel(r"$d\theta_E / dt$")
ax[0].set_title("Deterministic sweep used for kappa")
ax[0].legend(frameon=False)

ax[1].plot(theta_grid, c_be_theta, color="#7c3aed")
ax[1].set_xlabel(r"$\theta$")
ax[1].set_ylabel(r"$c_{BE}(\theta)$")
ax[1].set_title("Core target_diffusion profile")

plt.tight_layout()

calibration_summary = {
    "kappa": float(calibration_result["kappa"]),
    "c_be_theta_max": float(calibration_result["c_be_theta_max"]),
    "effective_c_be_max": float(calibration_result["effective_c_be_max"]),
    "certificate_passed": bool(calibration_result["certificate_passed"]),
}
calibration_summary


: 

## Step 2: one representative small-drift trajectory

This is the direct check the circuit should pass in the deterministic limit: `x_ref`, `x_E`, and `x_B` should overlap and remain close to linear after onset.


In [ ]:
example = run_trial(v=V_TARGET, seed=123, kappa=float(calibration_result["kappa"]))

fig, ax = plt.subplots(3, 1, figsize=(8.5, 7.0), sharex=True)
ax[0].plot(example["times"], example["x_ref"], color="#111827", label="reference x")
ax[0].plot(example["times"], example["x_E"], color="#1d4ed8", label="decoded x_E")
ax[0].plot(example["times"], example["x_B"], color="#059669", label="decoded x_B")
ax[0].axvline(T_START, color="#9ca3af", linestyle=":")
ax[0].set_ylabel("x")
ax[0].set_title("Deterministic drift trajectory")
ax[0].legend(frameon=False)

ax[1].plot(example["times"], example["theta_E"], color="#1d4ed8", label=r"$\theta_E$")
ax[1].plot(example["times"], example["theta_B"], color="#059669", label=r"$\theta_B$")
ax[1].axvline(T_START, color="#9ca3af", linestyle=":")
ax[1].set_ylabel(r"$\theta$")
ax[1].set_title("Edge-bump alignment")
ax[1].legend(frameon=False)

ax[2].plot(example["times"], example["v_drift"], color="#b45309", label=r"$v_{drift}$")
ax[2].plot(example["times"], example["v_drive"], color="#7c3aed", linestyle="--", label=r"$v_{drive}$")
ax[2].axvline(T_START, color="#9ca3af", linestyle=":")
ax[2].set_xlabel("time (ms)")
ax[2].set_ylabel("drive")
ax[2].set_title("Continuous scalar drive")
ax[2].legend(frameon=False)

plt.tight_layout()

single_trial_summary = {
    "v_target": float(example["v_target"]),
    "ref_slope": float(example["ref_slope"]),
    "edge_slope": float(example["edge_slope"]),
    "bump_slope": float(example["bump_slope"]),
    "mean_abs_error_xE": float(example["mean_abs_error_xE"]),
    "mean_abs_error_xB": float(example["mean_abs_error_xB"]),
    "alignment_mean_abs": float(example["alignment_mean_abs"]),
    "hit_boundary": bool(example["hit_boundary"]),
}
single_trial_summary


## Step 3: small drift sweep

This sweep checks the weak-drive regime more directly by comparing the fitted decoded slope against the reference slope from `x_traj`.


In [ ]:
trials = [run_trial(v=v, seed=200 + idx, kappa=float(calibration_result["kappa"])) for idx, v in enumerate(V_SWEEP)]
ref_slopes = np.array([trial["ref_slope"] for trial in trials], dtype=float)
edge_slopes = np.array([trial["edge_slope"] for trial in trials], dtype=float)
bump_slopes = np.array([trial["bump_slope"] for trial in trials], dtype=float)
edge_errors = np.array([trial["mean_abs_error_xE"] for trial in trials], dtype=float)
align_errors = np.array([trial["alignment_mean_abs"] for trial in trials], dtype=float)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].plot(V_SWEEP, ref_slopes, "o-", color="#111827", label="reference slope")
ax[0].plot(V_SWEEP, edge_slopes, "o-", color="#1d4ed8", label="edge decoded slope")
ax[0].plot(V_SWEEP, bump_slopes, "o-", color="#059669", label="bump decoded slope")
ax[0].set_xlabel("target drift v (x / s)")
ax[0].set_ylabel("fitted slope (x / ms)")
ax[0].set_title("Decoded slope tracks target slope")
ax[0].legend(frameon=False)

ax[1].plot(V_SWEEP, edge_errors, "o-", color="#7c3aed", label="mean |x_E - x_ref|")
ax[1].plot(V_SWEEP, align_errors, "s--", color="#b91c1c", label=r"mean $|\theta_B-\theta_E|$")
ax[1].set_xlabel("target drift v (x / s)")
ax[1].set_title("Weak-drive diagnostics")
ax[1].legend(frameon=False)

plt.tight_layout()

sweep_summary = {
    "v_sweep": [float(v) for v in V_SWEEP],
    "ref_slopes": [float(x) for x in ref_slopes],
    "edge_slopes": [float(x) for x in edge_slopes],
    "bump_slopes": [float(x) for x in bump_slopes],
    "mean_edge_error": float(np.mean(edge_errors)),
    "max_alignment_error": float(np.max(align_errors)),
    "any_boundary_hit": bool(any(trial["hit_boundary"] for trial in trials)),
}
sweep_summary


In [ ]:
final_report = {
    "calibration_summary": calibration_summary,
    "single_trial_summary": single_trial_summary,
    "sweep_summary": sweep_summary,
}
print(json.dumps(final_report, indent=2))


## Notes

- This notebook is the deterministic companion to the Brownian quick check.
- The public runtime path remains `target_diffusion`; the deterministic limit is just the special case `c = 0`.
- The next follow-up is to validate the combined case `dx = v \, dt + c \, dW` with both terms active at once.
